In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

In [ ]:
forced_blocked = 'me2d1.txt'
forced_interleaved = None
free = None

df_main = pd.read_json(forced_blocked, lines=True)
df_main

# Make sure scheduler works

In [ ]:
# Count number of practice and test trials
print('Number of trials in each stage')
display(df_main.groupby('stage').count().iloc[:, 1])

# Each feature pair must appear twice during practice and once during test (for each family)
df = df_main.filter(items=['stage', 'famInd', 'features'])
df['featuresStr'] = df.features.astype(str)
df = df.groupby(['stage', 'famInd', 'featuresStr']).count()

print('How many times each feature pair is presented in each family (and each stage)?')
display(df)

print('Are all features presented an equal number of times in each family?')
df = df.groupby(['stage', 'famInd']).nunique().eq(1)
display(df)

del df

# Make sure rules are correctly implemented

In [ ]:
# Check that feedback is given correctly
df = df_main.filter(items=['correctResponse', 'guess', 'correct'])
df['same'] = df.correctResponse == df.guess
print(f'Whenever the participant\'s response is the same as the true correct response, the trial is scored as correct: {np.all(df.same == df.correct)}')

# Check that the feature pairs map onto a their categories consistently
df = df_main.filter(items=['stage', 'famInd', 'features', 'correctResponse'])
df = df.join(df.features.apply(pd.Series).rename(columns={0: 'f1', 1: 'f2'})).drop(columns=['features'])
df['correctResponse'] = df.correctResponse.astype(str)
df['f1Jit'] = df.f1 + np.random.uniform(low=-.1, high=.1, size=df.shape[0])
df['f2Jit'] = df.f2 + np.random.uniform(low=-.1, high=.1, size=df.shape[0])
print('Plot below shows true correct response category for each family at each stage:')
with sns.color_palette("Set2"):
    sns.relplot(data=df, x='f1Jit', y='f2Jit', row='stage', col='famInd', hue='correctResponse')

del df


# Make sure response order is roughly equal

In [ ]:
# Check that response items appear on each side roughly equally often
df = df_main.filter(items=['stage', 'famInd', 'responseOrder', 'trialsComplete'])
df['responseOrderStr'] = df.responseOrder.astype(str)
display(df.groupby(['stage', 'famInd', 'responseOrderStr']).count())

del df

# Does confidence grow over time?

In [ ]:
df = df_main.filter(items=['stage', 'trialsComplete', 'famInd', 'correct'])
# df = df[df.stage.eq('free_epochs')]
df.head()
sns.lineplot(x='trialsComplete', y='correct', hue='famInd', data=df)

In [ ]:
def surprise(x, alpha):
    y = np.zeros_like(x)
    pos_dev = x - 0.5 > 0
    y[pos_dev] = alpha * np.sqrt(np.abs(x[pos_dev] - 0.5))
    y[~pos_dev] = np.abs(np.sqrt(-(x[~pos_dev] - 0.5)))
    return y

x = np.linspace(0, 1, 100)

ax = plt.gca()
ax.axvline(.5, color='k', ls='--')
ax.set_xlabel('Observed reward')
ax.set_ylabel('Noise coefficient')
ax.set_xlim(0, 1)

for alpha in range(-10, 10, 2):
    y = np.exp(1-surprise(x, alpha))
    ax.plot(x, y)

In [ ]:
def recency(t):
    return 1 - ((np.max(t) - t) / 25)

t = np.linspace(0, 25, 100)
y = recency(t)

ax = plt.gca()
ax.set_xlabel('Time since observation')
ax.set_ylabel('Noise coefficient')
# ax.set_xlim(0, 1)

plt.plot(t, np.exp(y))
